# Reproducibility Check — Federated Recommender System for Student Performance Prediction

This notebook independently verifies that the source code in
[rodrigotertulino/federated-recommender-education](https://github.com/rodrigoronner/federated-recommender-education)
faithfully implements the methodology described in the JEDM submission
*"Privacy-Preserving Personalization in Education: A Federated Recommender System for Student Performance Prediction"*
(Tertulino & Almeida, arXiv:2509.10516), and reports **actual results obtained by running the unmodified pipeline end-to-end**
against this dataset (`student-performance-for-recommender-systems`).

**What this notebook contains:**
1. A live, runnable version of the full pipeline (data loading → XGBoost baseline → federated FedAvg/FedProx via Flower) — a reduced-scale demo runs inline so the notebook executes end-to-end on Kaggle in a few minutes.
2. The **real, full-scale reproduction results** (100 communication rounds × 4 aggregation strategies, matching the paper's exact experimental setup) obtained by running this repository's code unmodified on a 10-core machine (~10h15min total runtime), embedded below as data so they render without external dependencies.
3. A transparent comparison against the paper's published Table 4-6, including where results matched closely and where they diverged.

## 0. Setup

`flwr` and `ray` are not preinstalled on Kaggle and are required for `flwr.simulation.start_simulation`.
This was an actual gap found in the repo's `requirements.txt` — `ray` was missing entirely — fixed for this notebook and reported upstream.

The repo's `requirements.txt` pins `flwr==1.7.0` / `ray==2.6.3` (tested with Python 3.10/3.11). Kaggle's current base image runs **Python 3.12**,
for which `ray==2.6.3` has no published wheel (`ray` only publishes py3.12 wheels from 2.31.0 onward) — pinning the repo's exact versions fails to install here.
This notebook therefore installs the latest `flwr`/`ray` (unpinned) instead; `flwr.simulation.start_simulation` remains available (deprecated in favor of `run_simulation`
but functional) and was verified to reproduce the same training behavior. `xgboost`/`torch`/`scikit-learn` are already preinstalled on Kaggle and are left untouched to avoid `numpy` ABI conflicts.

In [ ]:
!pip install -q flwr ray 2>&1 | tail -5

In [ ]:
import os, json, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
SUCCESS_THRESHOLD = 0.70  # matches Section 4.2, Step 3 of the paper

EXPECTED_PATH = '/kaggle/input/datasets/rodrigotertulino/student-performance-for-recommender-systems/interactions_real_rich_scaled_processed.csv'
if os.path.exists(EXPECTED_PATH):
    DATA_PATH = EXPECTED_PATH
else:
    print('Expected path not found, searching /kaggle/input ...')
    print('Top-level contents of /kaggle/input:', os.listdir('/kaggle/input') if os.path.isdir('/kaggle/input') else 'MISSING')
    candidates = glob.glob('/kaggle/input/**/*.csv', recursive=True)
    print('CSV files found:', candidates)
    assert candidates, 'No CSV found under /kaggle/input — is the dataset attached to this kernel (Add Data)?'
    DATA_PATH = candidates[0]
print('Using DATA_PATH =', DATA_PATH)

df = pd.read_csv(DATA_PATH)
df['target'] = (df['target_correct_rate'] >= SUCCESS_THRESHOLD).astype(int)

print('Shape:', df.shape)
print('Unique students:', df.user_id_new.nunique(), ' (paper reports 1,365)')
print('Unique skills  :', df.skill_id_new.nunique(), ' (paper reports 107)')
print(df['target'].value_counts(normalize=True).rename('class_share'))
df.head()

## 1. Centralized XGBoost Baseline (Section 4.3.1)

Same procedure as `02_centralized_baseline.py`: 80/20 stratified split (seed 42), then XGBoost refit for `n_estimators = 1..100`
to trace performance across boosting rounds, exactly as the paper's Figure 4 / Table 4.

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

FEATURES = ['user_id_new', 'skill_id_new', 'user_mean_correct', 'user_interaction_count', 'skill_mean_correct']
X, y = df[FEATURES].values, df['target'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=RANDOM_SEED, stratify=y)

records = []
model = None
for n in range(1, 101):
    model = xgb.XGBClassifier(n_estimators=n, eval_metric='logloss', random_state=RANDOM_SEED, verbosity=0)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    records.append({
        'round': n,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1_score': f1_score(y_test, y_pred, zero_division=0),
    })

xgb_metrics = pd.DataFrame(records)
best = xgb_metrics.loc[xgb_metrics.f1_score.idxmax()]
print(f"Peak F1={best.f1_score:.4f} at round {int(best['round'])} | Acc={best.accuracy:.4f} Prec={best.precision:.4f} Rec={best.recall:.4f}")

### Comparison against the paper (Table 4)

| Metric | Paper (Table 4) | This run |
|---|---|---|
| F1-Score  | 0.8285 (round 24) | *computed above* |
| Accuracy  | 0.7702 | *computed above* |
| Precision | 0.7919 | *computed above* |
| Recall    | 0.8686 | *computed above* |

In our independent full local run (10-core machine, same seed/config) we obtained **F1 = 0.8274 (round 20)**, Accuracy 0.7682, Precision 0.7890, Recall 0.8698 — within ~0.1pp of the paper across every metric.

## 2. Federated Learning — RecommenderNet + FedAvg/FedProx (Section 4.3.2)

`RecommenderNet` (2 embedding layers + 2 dense ReLU layers + sigmoid output, Table 3) trained via Flower, one client per student.
The FedProx proximal term `(mu/2)*||w - w_global||^2` is added to the local loss; `mu=0` recovers standard FedAvg exactly, matching the paper's stated correspondence.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

EMBEDDING_DIM, HIDDEN_1, HIDDEN_2, N_FEATURES = 10, 32, 16, 3

class StudentSkillDataset(Dataset):
    def __init__(self, d):
        self.user_ids = torch.tensor(d['user_id_new'].values, dtype=torch.long)
        self.skill_ids = torch.tensor(d['skill_id_new'].values, dtype=torch.long)
        self.features = torch.tensor(d[['user_mean_correct', 'user_interaction_count', 'skill_mean_correct']].values, dtype=torch.float32)
        self.labels = torch.tensor(d['target'].values, dtype=torch.float32)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return self.user_ids[i], self.skill_ids[i], self.features[i], self.labels[i]

class RecommenderNet(nn.Module):
    def __init__(self, num_users, num_skills):
        super().__init__()
        self.user_embedding = nn.Embedding(num_users, EMBEDDING_DIM)
        self.skill_embedding = nn.Embedding(num_skills, EMBEDDING_DIM)
        self.fc1 = nn.Linear(EMBEDDING_DIM * 2 + N_FEATURES, HIDDEN_1)
        self.fc2 = nn.Linear(HIDDEN_1, HIDDEN_2)
        self.output = nn.Linear(HIDDEN_2, 1)
        self.relu, self.sigmoid = nn.ReLU(), nn.Sigmoid()
    def forward(self, u, s, f):
        x = torch.cat([self.user_embedding(u), self.skill_embedding(s), f], dim=1)
        x = self.relu(self.fc1(x)); x = self.relu(self.fc2(x))
        return self.sigmoid(self.output(x)).squeeze(-1)
    def get_parameters_flat(self): return [p.detach().cpu().numpy() for p in self.parameters()]
    def set_parameters_flat(self, params):
        state = self.state_dict()
        for k, a in zip(state.keys(), params): state[k] = torch.tensor(a)
        self.load_state_dict(state, strict=True)

def train_local(model, dataset, epochs=5, batch_size=32, lr=1e-3, mu=0.0, global_params=None, device='cpu'):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.999), eps=1e-8)
    criterion = nn.BCELoss()
    model.to(device); model.train()
    if mu > 0 and global_params is not None:
        global_tensors = [torch.tensor(p, device=device) for p in global_params]
    for _ in range(epochs):
        for u, s, f, lab in loader:
            u, s, f, lab = u.to(device), s.to(device), f.to(device), lab.to(device)
            optimizer.zero_grad()
            preds = model(u, s, f)
            loss = criterion(preds, lab)
            if mu > 0 and global_params is not None:
                prox = sum(torch.norm(p - g) ** 2 for p, g in zip(model.parameters(), global_tensors))
                loss = loss + (mu / 2.0) * prox
            loss.backward(); optimizer.step()
    return model.get_parameters_flat(), len(dataset)

def evaluate_local(model, dataset, batch_size=64, device='cpu'):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    model.to(device); model.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for u, s, f, lab in loader:
            preds = model(u.to(device), s.to(device), f.to(device))
            preds_all.extend((preds.cpu().numpy() >= 0.5).astype(int).tolist())
            labels_all.extend(lab.numpy().astype(int).tolist())
    return {'accuracy': accuracy_score(labels_all, preds_all), 'precision': precision_score(labels_all, preds_all, zero_division=0),
            'recall': recall_score(labels_all, preds_all, zero_division=0), 'f1_score': f1_score(labels_all, preds_all, zero_division=0)}

### Quick live demo

The paper's exact setup (100 rounds x 4 strategies, 1,365 simulated clients) took **~10h15min** end-to-end on a 10-core machine — Flower's virtual-client actor pool overhead dominates since Kaggle sessions have fewer cores and a runtime budget. 
To keep this notebook runnable end-to-end on Kaggle in a few minutes while still proving the code executes correctly, this cell runs a **reduced-scale** demo (fewer rounds, single strategy). 
Section 3 below reports the **real full-scale (100-round x 4-strategy) results**, obtained by running this exact code with `QUICK_DEMO=False` offline.

In [ ]:
import flwr as fl
from flwr.client import NumPyClient
from flwr.common import ndarrays_to_parameters
from flwr.server.strategy import FedProx

QUICK_DEMO = True          # set False (and N_ROUNDS=100, run all 4 mu values) to reproduce the paper exactly — expect several hours
N_ROUNDS = 5 if QUICK_DEMO else 100
DEMO_MU = 0.5               # the paper's optimal configuration
LOCAL_EPOCHS, LOCAL_BATCH_SIZE, LEARNING_RATE = 5, 32, 1e-3
FRACTION_FIT, FRACTION_EVAL, MIN_FIT_CLIENTS, VAL_FRACTION = 0.10, 0.20, 50, 0.20
DEVICE = 'cpu'
torch.manual_seed(RANDOM_SEED)

NUM_USERS, NUM_SKILLS = df.user_id_new.nunique(), df.skill_id_new.nunique()
client_train, client_val = {}, {}
for uid, g in df.groupby('user_id_new'):
    if len(g) >= 2:
        tr, va = train_test_split(g, test_size=VAL_FRACTION, random_state=RANDOM_SEED, shuffle=True)
    else:
        tr, va = g, g.iloc[0:0]
    client_train[uid] = StudentSkillDataset(tr.reset_index(drop=True))
    client_val[uid]   = StudentSkillDataset(va.reset_index(drop=True))
N_CLIENTS = len(client_train)

class StudentClient(NumPyClient):
    def __init__(self, uid):
        self.uid = uid
        self.model = RecommenderNet(NUM_USERS, NUM_SKILLS)
    def get_parameters(self, config): return self.model.get_parameters_flat()
    def fit(self, parameters, config):
        self.model.set_parameters_flat(parameters)
        mu = float(config.get('proximal_mu', 0.0))
        params, n = train_local(self.model, client_train[self.uid], epochs=LOCAL_EPOCHS, batch_size=LOCAL_BATCH_SIZE,
                                 lr=LEARNING_RATE, mu=mu, global_params=parameters if mu > 0 else None, device=DEVICE)
        return params, n, {}
    def evaluate(self, parameters, config):
        self.model.set_parameters_flat(parameters)
        val_ds = client_val[self.uid]
        if len(val_ds) == 0:
            return 0.0, 0, {'accuracy': 0.0, 'precision': 0.0, 'recall': 0.0, 'f1_score': 0.0}
        metrics = evaluate_local(self.model, val_ds, device=DEVICE)
        return 0.0, len(val_ds), metrics

def client_fn(cid): return StudentClient(uid=int(cid)).to_client()

def weighted_average(metrics):
    total = sum(n for n, _ in metrics if n > 0)
    if total == 0: return {'accuracy': 0.0, 'precision': 0.0, 'recall': 0.0, 'f1_score': 0.0}
    return {k: sum(n * m[k] for n, m in metrics if n > 0) / total for k in ['accuracy', 'precision', 'recall', 'f1_score']}

def fit_config(server_round): return {'local_epochs': LOCAL_EPOCHS, 'batch_size': LOCAL_BATCH_SIZE, 'lr': LEARNING_RATE}

strategy = FedProx(fraction_fit=FRACTION_FIT, fraction_evaluate=FRACTION_EVAL, min_fit_clients=MIN_FIT_CLIENTS,
                    min_evaluate_clients=2, min_available_clients=N_CLIENTS, on_fit_config_fn=fit_config,
                    evaluate_metrics_aggregation_fn=weighted_average,
                    initial_parameters=ndarrays_to_parameters(RecommenderNet(NUM_USERS, NUM_SKILLS).get_parameters_flat()),
                    proximal_mu=DEMO_MU)

history = fl.simulation.start_simulation(client_fn=client_fn, num_clients=N_CLIENTS,
                                          config=fl.server.ServerConfig(num_rounds=N_ROUNDS), strategy=strategy,
                                          client_resources={'num_cpus': 1}, ray_init_args={'include_dashboard': False, 'ignore_reinit_error': True})

dist = history.metrics_distributed
demo_df = pd.DataFrame([{'round': r, **{k: dict(dist[k])[r] for k in ['accuracy', 'precision', 'recall', 'f1_score']}} for r, _ in dist['f1_score']])
print(f'Live demo ({N_ROUNDS} rounds, mu={DEMO_MU}) — final round metrics:')
demo_df.tail()

## 3. Full-Scale Reproduction Results (100 rounds x 4 strategies)

The data embedded below is the **unmodified, real output** of running `03_federated_training.py --all` from the repository
(100 communication rounds x {FedAvg, FedProx mu=0.1, FedProx mu=0.5, FedProx mu=1.0}) against this dataset — a ~10h15min run on a
10-core machine. It is embedded as CSV text so this section renders correctly without needing external file attachments.

In [ ]:
import io
centralized_full = pd.read_csv(io.StringIO('''round,accuracy,precision,recall,f1_score
1,0.708,0.6912512716174974,0.9814636494944632,0.8111818543573418
2,0.7416923076923077,0.7303182579564489,0.9446316803081367,0.8237640390469193
3,0.7464615384615385,0.7415188897455667,0.9260953298025999,0.8235923785056734
4,0.7567692307692307,0.7592182147894418,0.9070775156475686,0.8265876933201711
5,0.7596923076923077,0.7744599745870394,0.8803562831006259,0.82401982875169
6,0.7601538461538462,0.7765934768706033,0.8769860375541647,0.8237422272470323
7,0.7607692307692308,0.7801250269454624,0.8712084737602311,0.8231547822131241
8,0.7621538461538462,0.7818849978383052,0.8707270101107366,0.8239179954441913
9,0.7624615384615384,0.7827123050259965,0.8697640828117477,0.823945267958951
10,0.7630769230769231,0.7829004329004329,0.8707270101107366,0.8244814223843173
11,0.764,0.7840416305290546,0.8704862782859895,0.8250057038558065
12,0.7643076923076924,0.7852480417754569,0.8688011555127588,0.8249142857142857
13,0.764,0.7851545494122769,0.8683196918632643,0.8246456332876086
14,0.764,0.7852787456445993,0.8680789600385171,0.824605533958381
15,0.7646153846153846,0.7862129144851658,0.8675974963890226,0.8249027237354085
16,0.7649230769230769,0.7865560890440856,0.8675974963890226,0.825091575091575
17,0.7650769230769231,0.7866026620117826,0.8678382282137699,0.8252260501316241
18,0.7650769230769231,0.7859786631831047,0.869041887337506,0.8254258602949582
19,0.7666153846153846,0.787819253438114,0.8688011555127588,0.8263308528906698
20,0.7681538461538462,0.7890369076217515,0.8697640828117477,0.8274361616855606
21,0.7669230769230769,0.7882892724492025,0.8685604236880116,0.8264803573473829
22,0.7684615384615384,0.7900153273483688,0.8685604236880116,0.8274280472422887
23,0.7684615384615384,0.7900153273483688,0.8685604236880116,0.8274280472422887
24,0.7675384615384615,0.790503407342273,0.8656716417910447,0.8263817074571986
25,0.7678461538461538,0.7908511106223884,0.8656716417910447,0.8265716584300655
26,0.7672307692307693,0.7902835788085294,0.8654309099662976,0.8261519016431116
27,0.7670769230769231,0.7902374670184696,0.8651901781415503,0.8260170075844634
28,0.7678461538461538,0.7911071978868589,0.8651901781415503,0.8264918937564677
29,0.7675384615384615,0.7907590759075908,0.8651901781415503,0.8263018737785952
30,0.7683076923076924,0.7917584839136184,0.864949446316803,0.8267372296364474
31,0.7681538461538462,0.7920988744206577,0.8639865190178142,0.8264824409902131
32,0.7681538461538462,0.7920988744206577,0.8639865190178142,0.8264824409902131
33,0.7673846153846153,0.7914827890556045,0.8635050553683197,0.825926778724384
34,0.7673846153846153,0.7916114790286976,0.8632643235435724,0.8258866881621373
35,0.7670769230769231,0.7913907284768212,0.8630235917188253,0.8256563795485952
36,0.7672307692307693,0.7915654669905057,0.8630235917188253,0.8257514683865024
37,0.7675384615384615,0.7898661987277912,0.8668753009147809,0.8265809709629289
38,0.7667692307692308,0.7890008764241893,0.8668753009147809,0.8261069052534985
39,0.7667692307692308,0.7890008764241893,0.8668753009147809,0.8261069052534985
40,0.7664615384615384,0.7900528169014085,0.8642272508425614,0.8254771211772821
41,0.7663076923076924,0.7902621722846442,0.8635050553683197,0.8252617048199701
42,0.7666153846153846,0.790610535596209,0.8635050553683197,0.825451616614889
43,0.7669230769230769,0.7913446676970634,0.862782859894078,0.8255211332488771
44,0.766,0.7910678753040018,0.8613384689455946,0.8247090008067305
45,0.7663076923076924,0.7912889674994472,0.8615792007703419,0.824939495217241
46,0.7658461538461538,0.7908930150309461,0.8613384689455946,0.8246139663516939
47,0.7661538461538462,0.7912428129146395,0.8613384689455946,0.8248040571692024
48,0.7658461538461538,0.7912793271359009,0.8606162734713529,0.8244926199261993
49,0.7663076923076924,0.7916758910781492,0.8608570052961001,0.8248183600507438
50,0.7655384615384615,0.7914450354609929,0.859653346172364,0.8241403184860374
51,0.7655384615384615,0.7913159060700045,0.8598940779971113,0.8241808952468851
52,0.7650769230769231,0.7905330679053307,0.8603755416466057,0.8239769452449568
53,0.7655384615384615,0.7915742793791575,0.8594126143476167,0.8240997229916898
54,0.7663076923076924,0.7923230530286222,0.859653346172364,0.8246160951391295
55,0.7667692307692308,0.7929808973789427,0.8594126143476167,0.8248613678373382
56,0.7676923076923077,0.7935168738898757,0.8603755416466057,0.8255948255948256
57,0.7686153846153846,0.7944444444444444,0.8606162734713529,0.8262075340882828
58,0.7684615384615384,0.7941372418387741,0.8608570052961001,0.8261522467367448
59,0.768,0.7938693913816082,0.8603755416466057,0.8257855822550831
60,0.7666153846153846,0.7931954636424283,0.858690418873375,0.8246445497630331
61,0.766,0.7924905576538547,0.858690418873375,0.824263431542461
62,0.7658461538461538,0.7924444444444444,0.8584496870486278,0.8241275710654032
63,0.7658461538461538,0.7924444444444444,0.8584496870486278,0.8241275710654032
64,0.7661538461538462,0.7926666666666666,0.858690418873375,0.8243586780679455
65,0.7664615384615384,0.7932799287939475,0.8582089552238806,0.824468085106383
66,0.7666153846153846,0.7933259176863181,0.8584496870486278,0.8246040004624812
67,0.766,0.792880978865406,0.8579682233991334,0.8241415192507804
68,0.7661538461538462,0.7927967985771454,0.8584496870486278,0.8243180767452613
69,0.766,0.7932724437513923,0.8572460279248917,0.8240194376952447
70,0.766,0.7931418392340236,0.857486759749639,0.8240601503759398
71,0.7653846153846153,0.7930880713489409,0.8562831006259027,0.8234749392290774
72,0.7656923076923077,0.7930496769881934,0.8570052961001444,0.8237880365613791
73,0.7649230769230769,0.7925579322638147,0.8562831006259027,0.8231890766026383
74,0.764,0.7918894830659536,0.855560905151661,0.8224947928720203
75,0.7632307692307693,0.791657372295338,0.8543572460279248,0.8218131295588746
76,0.7633846153846154,0.7918340026773761,0.8543572460279248,0.8219082908754053
77,0.7629230769230769,0.7914343073834486,0.8541165142031777,0.8215815676739608
78,0.7632307692307693,0.7920482465937011,0.8536350505536832,0.821689259645464
79,0.7627692307692308,0.7919088064371926,0.8529128550794415,0.8212795549374131
80,0.7632307692307693,0.7920482465937011,0.8536350505536832,0.821689259645464
81,0.7638461538461538,0.7923643670462157,0.8543572460279248,0.8221939071006602
82,0.7643076923076924,0.7926339285714286,0.8548387096774194,0.822561964327079
83,0.7638461538461538,0.7923643670462157,0.8543572460279248,0.8221939071006602
84,0.7638461538461538,0.7923643670462157,0.8543572460279248,0.8221939071006602
85,0.7652307692307693,0.7935656836461126,0.8550794415021666,0.8231749710312862
86,0.7650769230769231,0.7935195530726257,0.8548387096774194,0.8230385907984703
87,0.7650769230769231,0.7935195530726257,0.8548387096774194,0.8230385907984703
88,0.7647692307692308,0.7930341594105828,0.8550794415021666,0.8228889146299085
89,0.7646153846153846,0.7928571428571428,0.8550794415021666,0.8227936066712995
90,0.7652307692307693,0.7934345690040197,0.8553201733269138,0.8232159406858202
91,0.7653846153846153,0.7936117936117936,0.8553201733269138,0.8233113196616846
92,0.7649230769230769,0.7929495760821063,0.855560905151661,0.8230662343677628
93,0.7643076923076924,0.7923728813559322,0.8553201733269138,0.8226441305857838
94,0.7649230769230769,0.7928189116859946,0.8558016369764083,0.8231072007409123
95,0.7658461538461538,0.7933571110120374,0.8567645642753972,0.8238425925925926
96,0.7646153846153846,0.7924654480606331,0.8558016369764083,0.8229166666666666
97,0.7641538461538462,0.7924570408390984,0.8548387096774194,0.822466705269253
98,0.7638461538461538,0.7922338763668824,0.8545979778526721,0.8222350897510133
99,0.7633846153846154,0.792225201072386,0.8536350505536832,0.821784472769409
100,0.7644615384615384,0.7937290033594625,0.8531535869041887,0.8223691843601346'''))

In [ ]:
import io
fedavg_full = pd.read_csv(io.StringIO('''round,accuracy,precision,recall,f1_score
1,0.6414653156664069,0.6414653156664069,0.921278254091972,0.7356187316831551
2,0.6396957123098201,0.6396957123098201,0.9343015214384509,0.737546813234037
3,0.6264975334742776,0.6264975334742776,0.923185341789993,0.7279636090309324
4,0.6149812734082397,0.6149812734082397,0.9123595505617977,0.7127463484578639
5,0.6229739252995067,0.6229739252995067,0.9055673009161381,0.7206491301333948
6,0.6102305475504323,0.6102305475504323,0.9070605187319885,0.709458250000085
7,0.6260043827611396,0.6260043827611396,0.9130752373995618,0.7234948596006586
8,0.622511485451761,0.622511485451761,0.9127105666156202,0.7193703146221336
9,0.6569069069069069,0.6569069069069069,0.9144144144144144,0.7463974825752271
10,0.6151537884471118,0.6151537884471118,0.9069767441860465,0.7108468367082965
11,0.6515994436717664,0.6515994436717664,0.9325452016689847,0.7456997987800341
12,0.6389078498293516,0.6389078498293516,0.9351535836177475,0.7399871046720747
13,0.6147473073736537,0.6147473073736537,0.9163214581607291,0.7118760242153663
14,0.6468885672937771,0.6468885672937771,0.9211287988422576,0.7407259732198603
15,0.6323308270676692,0.6323308270676692,0.9195488721804511,0.7261295581985217
16,0.6456808199121523,0.6456808199121523,0.9231332357247438,0.7383565338860548
17,0.6060606060606061,0.6060606060606061,0.9024390243902439,0.7033178496520229
18,0.6521739130434783,0.6521739130434783,0.9312762973352033,0.750242869080443
19,0.6252676659528907,0.6252676659528907,0.8872234118486795,0.7153319856613445
20,0.625,0.625,0.9210526315789473,0.7233893928289827
21,0.6636553161917998,0.6636553161917998,0.9381514940931203,0.7586270777633998
22,0.6324029952348537,0.6324029952348537,0.9189925119128659,0.7290783814603806
23,0.6819739547635366,0.6819739547635366,0.9410555174777244,0.7710291566564039
24,0.6422413793103449,0.6422413793103449,0.9274425287356322,0.7384050061422873
25,0.6240928882438317,0.6240928882438317,0.895500725689405,0.7182143873985096
26,0.6264450867052023,0.6264450867052023,0.9161849710982659,0.7207218092570513
27,0.6134728580771747,0.6134728580771747,0.9215173315892741,0.7153752235151262
28,0.6469807564698076,0.6469807564698076,0.9256801592568016,0.7415924876669522
29,0.6378227494766224,0.6378227494766224,0.9239357990230286,0.7326206988920574
30,0.6414401175606171,0.6414401175606171,0.9265246142542248,0.7353724417707297
31,0.6164280331574982,0.6164280331574982,0.9118311981914092,0.7148646735418703
32,0.6306179775280899,0.6306179775280899,0.9199438202247191,0.7263087919482608
33,0.6103321033210332,0.6103321033210332,0.9099630996309963,0.7118865552212258
34,0.6304347826086957,0.6304347826086957,0.9291725105189341,0.7310119943688882
35,0.64280798348245,0.64280798348245,0.9318651066758431,0.7386386953556812
36,0.6513761467889908,0.6513761467889908,0.9357798165137615,0.7458033247074943
37,0.6217423678332092,0.6217423678332092,0.9233060312732688,0.7222850926133338
38,0.6250962278675904,0.6250962278675904,0.9160892994611239,0.7194745976105384
39,0.5942684766214178,0.5942684766214178,0.9019607843137255,0.6938785811553333
40,0.6065798492117889,0.6065798492117889,0.9191226867717615,0.7093092047071107
41,0.6359680928208847,0.6359680928208847,0.9151559100797679,0.7285646057071345
42,0.6159420289855072,0.6159420289855072,0.9188405797101449,0.7177244285452058
43,0.588865096359743,0.588865096359743,0.9157744468236974,0.6937091485231822
44,0.6413119330076762,0.6413119330076762,0.9239357990230286,0.7358255587753237
45,0.6295764536970567,0.6295764536970567,0.9267767408470926,0.7304932209889458
46,0.6376917457998539,0.6376917457998539,0.902118334550767,0.7297015893283125
47,0.6454413892908828,0.6454413892908828,0.9341534008683068,0.7419599863835713
48,0.6431686046511628,0.6431686046511628,0.9302325581395349,0.7404371015994388
49,0.6396790663749088,0.6396790663749088,0.9241429613420861,0.7360849120574086
50,0.6296561604584527,0.6296561604584527,0.9212034383954155,0.7269650972126114
51,0.6400580551523948,0.6400580551523948,0.9136429608127722,0.732493060117842
52,0.6049552649690296,0.6049552649690296,0.9098417068134893,0.7060279705837923
53,0.6262203626220363,0.6262203626220363,0.9323570432357043,0.7270458789916401
54,0.6430615164520744,0.6430615164520744,0.932761087267525,0.7394600168670139
55,0.6264726264726265,0.6264726264726265,0.9175329175329175,0.7232011932816039
56,0.6471009305654974,0.6471009305654974,0.9234073013600572,0.7398995579627091
57,0.6373873873873874,0.6373873873873874,0.9151651651651652,0.7299261355389177
58,0.6207906295754027,0.6207906295754027,0.9260614934114202,0.721882507459414
59,0.6691127418278853,0.6691127418278853,0.9392928619079386,0.7642517548923522
60,0.6490963855421686,0.6490963855421686,0.9246987951807228,0.7447611341922993
61,0.6440198159943383,0.6440198159943383,0.9115357395612173,0.7329789157480753
62,0.637233259749816,0.637233259749816,0.9286239882266373,0.7333885655226007
63,0.643490115882754,0.643490115882754,0.9372869802317655,0.7424201197753353
64,0.6445255474452555,0.6445255474452555,0.9313868613138686,0.7435634761839991
65,0.6210746685275645,0.6210746685275645,0.9351011863224006,0.7253485829302669
66,0.6427576601671309,0.6427576601671309,0.9345403899721448,0.7410328522299162
67,0.61863173216885,0.61863173216885,0.9170305676855895,0.7192226259538402
68,0.6448398576512455,0.6448398576512455,0.9238434163701068,0.73950243651185
69,0.6161473087818697,0.6161473087818697,0.9298866855524079,0.7204604871868809
70,0.6464263124604681,0.6464263124604681,0.9354838709677419,0.7454549881505088
71,0.6268871315600287,0.6268871315600287,0.9237958303378864,0.7237134804560934
72,0.6015209125475285,0.6015209125475285,0.9072243346007605,0.7003188022076152
73,0.6604426002766252,0.6604426002766252,0.9467496542185339,0.7583229539498302
74,0.6351056081573198,0.6351056081573198,0.9184268026219956,0.7312820362366135
75,0.6319241982507289,0.6319241982507289,0.9307580174927114,0.7341062470299914
76,0.6331621423330888,0.6331621423330888,0.9310344827586207,0.7332281519568449
77,0.618380062305296,0.618380062305296,0.9182242990654206,0.717374330180147
78,0.5893566322478158,0.5893566322478158,0.9118347895154885,0.6939000305467972
79,0.6320621468926554,0.6320621468926554,0.9074858757062146,0.7263695273446253
80,0.6288933068257124,0.6288933068257124,0.9304174950298211,0.7295794498861042
81,0.6089299461123941,0.6089299461123941,0.9068514241724404,0.7044918424738689
82,0.612094395280236,0.6117256637168141,0.9365781710914455,0.7143617358370304
83,0.6302465023317788,0.6302465023317788,0.9253830779480346,0.7306240144024566
84,0.6158707865168539,0.6158707865168539,0.9220505617977528,0.719306510657757
85,0.6222071767095464,0.6222071767095464,0.933649289099526,0.7252954048751694
86,0.63003663003663,0.6300704423781347,0.9222222222222224,0.7268882458252801
87,0.6304508499630451,0.6297970322360567,0.9364375461936437,0.733648871776621
88,0.6607142857142857,0.6603571428571429,0.9335714285714286,0.752821104496015
89,0.6204735376044568,0.620636025998143,0.9141132776230269,0.7179526330161089
90,0.6530314097881665,0.6530314097881665,0.9349890430971513,0.7484049238203768
91,0.6180392156862745,0.6183380018674136,0.9150326797385622,0.716789725710561
92,0.6111111111111112,0.6106648817492192,0.9250334672021419,0.714013317825554
93,0.640495867768595,0.6397743670470944,0.9192837465564738,0.7343784521306503
94,0.6280530355896721,0.6285681055394942,0.9097464526634101,0.7223306817490438
95,0.5847033595425304,0.5852649851935056,0.9177984274481773,0.6918727457838056
96,0.625090645395214,0.6267093131668912,0.9312476259539348,0.7244655878494964
97,0.6177285318559557,0.6170648661126501,0.9106648199445984,0.7152624507161723
98,0.6263194933145672,0.6267986997754766,0.9114025547523084,0.7224503561581331
99,0.6299529885829416,0.6300809108062299,0.932700917841952,0.7322106012783248
100,0.6340097970608818,0.634343030424206,0.9196862153475168,0.7286538028735003'''))

In [ ]:
import io
fedprox01_full = pd.read_csv(io.StringIO('''round,accuracy,precision,recall,f1_score
1,0.6582446808510638,0.6582446808510638,0.9394946808510638,0.7517993882145203
2,0.6413793103448275,0.6413793103448275,0.9337931034482758,0.7394505888502085
3,0.6164772727272727,0.6164772727272727,0.9232954545454546,0.7187077965370269
4,0.6357850070721358,0.6357850070721358,0.9306930693069307,0.7335221020712082
5,0.6136363636363636,0.6136363636363636,0.9265151515151515,0.7149939396554159
6,0.6314623338257016,0.6314623338257016,0.9172821270310192,0.7276054081685047
7,0.5967623252391464,0.5967623252391464,0.9050772626931567,0.6976224767502699
8,0.6420416386836804,0.6420416386836804,0.9442578912021491,0.740538028230404
9,0.6528685548293391,0.6528685548293391,0.9273783587509078,0.746499861154586
10,0.6143344709897611,0.6143344709897611,0.9262798634812287,0.7194055938972204
11,0.6194444444444445,0.6194444444444445,0.91875,0.7198453300433312
12,0.6077712609970675,0.6077712609970675,0.9281524926686217,0.7132980443563408
13,0.6347826086956522,0.6347826086956522,0.9282608695652174,0.7308594620859576
14,0.6158833063209076,0.6158833063209076,0.9100486223662885,0.7134144583760212
15,0.6069615663524293,0.6069615663524293,0.9216823785351704,0.7103910725265536
16,0.6369518332135155,0.6369518332135155,0.918763479511143,0.731193017988461
17,0.6422018348623854,0.6422018348623854,0.9336626676076217,0.7421143994877702
18,0.6387575962187712,0.6387575962187712,0.9237002025658338,0.7340773267906845
19,0.5990404386566142,0.5990404386566142,0.9287183002056203,0.7029365744873489
20,0.6365475387727579,0.6365475387727579,0.9318948078219824,0.7376187182302859
21,0.6322368421052632,0.6322368421052632,0.9296052631578947,0.7297150412969674
22,0.6377622377622377,0.6377622377622377,0.9188811188811189,0.7335196663203293
23,0.6477900552486188,0.6477900552486188,0.9205801104972375,0.7397737043298737
24,0.6125461254612546,0.6125461254612546,0.9129151291512915,0.7106058157338295
25,0.6071953010279001,0.6071953010279001,0.9140969162995595,0.7067165948397656
26,0.6152841280209014,0.6152841280209014,0.9340300457217505,0.7213367791475728
27,0.6086626139817629,0.6086626139817629,0.8951367781155015,0.7017962518652746
28,0.6238872403560831,0.6238872403560831,0.922106824925816,0.7240643793157576
29,0.6358873608382449,0.6358873608382449,0.9332023575638507,0.7364258234086258
30,0.6249089584850692,0.6249089584850692,0.9162418062636563,0.7191380427045727
31,0.6354515050167224,0.6354515050167224,0.9311036789297659,0.7329826912061466
32,0.6608344549125168,0.6608344549125168,0.9448183041722745,0.7571665473707405
33,0.6288659793814433,0.6288659793814433,0.9086892488954345,0.721795618066847
34,0.6437275985663082,0.6437275985663082,0.9304659498207886,0.7395924762943784
35,0.6153324287652646,0.6153324287652646,0.9287652645861602,0.7171103683934008
36,0.6295793758480326,0.6295793758480326,0.9226594301221167,0.7305351863407104
37,0.667524115755627,0.667524115755627,0.9453376205787781,0.7637523060261131
38,0.6148202494497432,0.6148202494497432,0.9097578870139399,0.7114561718244389
39,0.6337126600284495,0.6337126600284495,0.9238975817923186,0.7314583979094943
40,0.6200417536534447,0.6200417536534447,0.9157967988865693,0.7189664352543325
41,0.626188734455011,0.626188734455011,0.9246525237746891,0.7249122867719575
42,0.6306695464362851,0.6306695464362851,0.9222462203023758,0.7265947404690998
43,0.6126126126126126,0.6126126126126126,0.9147609147609148,0.7152236851370601
44,0.6108757062146892,0.6108757062146892,0.9371468926553672,0.7150919788604927
45,0.6369137670196672,0.6369137670196672,0.9236006051437217,0.7343731111871958
46,0.6241084165477889,0.6241084165477889,0.9172610556348074,0.7221436133234603
47,0.6243798724309001,0.6243798724309001,0.9241672572643516,0.7235719845856928
48,0.6556701030927835,0.6556701030927835,0.9429553264604811,0.7542287018162203
49,0.6266094420600858,0.6266094420600858,0.9241773962804005,0.7251521228062969
50,0.6149116064565718,0.6149116064565718,0.8985395849346657,0.709886817061122
51,0.5810276679841897,0.5810276679841897,0.908300395256917,0.6865420585106801
52,0.628755364806867,0.628755364806867,0.9098712446351931,0.7230047412512991
53,0.6474872541879096,0.6474872541879096,0.9293517844136926,0.741972947151136
54,0.6483825597749648,0.6483825597749648,0.9205344585091421,0.740101298874418
55,0.6248348745046235,0.6248348745046235,0.9326287978863936,0.7268310871801263
56,0.6132478632478633,0.6132478632478633,0.9081196581196581,0.7121198880830474
57,0.642083897158322,0.642083897158322,0.9343707713125846,0.7409030144145573
58,0.633883704235463,0.633883704235463,0.9138549892318737,0.727147686818554
59,0.6478779840848806,0.6478779840848806,0.9323607427055703,0.7451190125597507
60,0.6259097525473072,0.6259097525473072,0.9264919941775837,0.7238646886763761
61,0.6473867595818815,0.6473867595818815,0.9310104529616725,0.7436669161202443
62,0.6203821656050955,0.6203821656050955,0.9248407643312102,0.7240383979004619
63,0.6592890677397719,0.6592890677397719,0.9282360831656606,0.7509278958873769
64,0.6318883174136665,0.6318883174136665,0.9132990448199853,0.7223603029145875
65,0.5801063022019742,0.5801063022019742,0.9005315110098709,0.6831224229517502
66,0.6334332833583208,0.6334332833583208,0.9265367316341829,0.7323592550817748
67,0.6324614352783366,0.6324614352783366,0.9094567404426559,0.7274075976461781
68,0.6744847192608386,0.6744847192608386,0.9310589907604833,0.7617946010245858
69,0.6186206896551724,0.6186206896551724,0.9193103448275862,0.7189110223519235
70,0.652046783625731,0.652046783625731,0.9049707602339181,0.7399170514754551
71,0.6314344543582704,0.6314344543582704,0.9272477693891558,0.7331260832076754
72,0.6348973607038123,0.6348973607038123,0.9237536656891495,0.73249420748379
73,0.6423658872077029,0.6423658872077029,0.937414030261348,0.7435573739793679
74,0.6271899088997898,0.6271899088997898,0.9243167484232656,0.7290137215539059
75,0.6059678653404744,0.6059678653404744,0.9227237949502678,0.7081200069461124
76,0.6210884353741497,0.6210884353741497,0.9285714285714286,0.7231378554524974
77,0.6614397719173201,0.6614397719173201,0.9315751960085531,0.7570435282773705
78,0.6554681915654038,0.6554681915654038,0.9363831308077198,0.7498532978063024
79,0.6401408450704226,0.6406103286384977,0.9363849765258215,0.7395557358918315
80,0.6178736517719569,0.6178736517719569,0.8998459167950693,0.7121727364539694
81,0.6616979909267661,0.6626784319590541,0.9309786130913805,0.7558597743973956
82,0.642664872139973,0.6427770300583222,0.933153880663975,0.7395883388235934
83,0.6381350034794712,0.6378673518548258,0.9331941544885177,0.73932569709905
84,0.6361130254996554,0.6374798989202849,0.9293631649765351,0.7356483525537202
85,0.6257267441860465,0.6256904069767443,0.8987749169435216,0.7196363862506465
86,0.6205813040062844,0.6201361612987694,0.9159465828750982,0.7153065652836215
87,0.6653225806451613,0.6669466845878136,0.9317076292882744,0.7564075585679664
88,0.6237046632124352,0.6236945804627152,0.9240069084628669,0.7230983770942989
89,0.6418732782369146,0.6416150137741047,0.9387913223140496,0.7385426786521936
90,0.6299589603283173,0.6307349057691055,0.9245118766322595,0.7293443708281907
91,0.5984308131241084,0.5982202296039671,0.9091351765210539,0.6999027700097092
92,0.6095505617977528,0.6118518146959158,0.9090679919331603,0.7119318646569728
93,0.6325016857720837,0.633644532211626,0.9370440752773861,0.7346995358766674
94,0.6126510305614783,0.6132653061224489,0.8974267854864869,0.7066706869527346
95,0.6170542635658914,0.6189739803693292,0.8793421694584486,0.705730414326895
96,0.6067164179104477,0.6077090508993493,0.8909147320341351,0.7022103128723267
97,0.6598079561042524,0.6657685021882553,0.9267635542326901,0.7521314956882201
98,0.6335820895522388,0.6374037778142255,0.8879732518725055,0.7205778525073544
99,0.6209952283571916,0.623735031914991,0.909342350446645,0.7162767345631882
100,0.6289262235208181,0.6319359073731093,0.9048662562176076,0.7242559979908642'''))

In [ ]:
import io
fedprox05_full = pd.read_csv(io.StringIO('''round,accuracy,precision,recall,f1_score
1,0.6672991777356104,0.6672991777356104,0.9392789373814042,0.7631043920125686
2,0.621978021978022,0.621978021978022,0.9098901098901099,0.7181597571104815
3,0.6458625525946704,0.6458625525946704,0.9368863955119214,0.7436132626280328
4,0.6388101983002833,0.6388101983002833,0.9362606232294618,0.7360821484681147
5,0.6471816283924844,0.6471816283924844,0.9366736256089074,0.7458185212428626
6,0.6190803019903912,0.6190803019903912,0.9210706932052162,0.7210708627910828
7,0.612037708484409,0.612037708484409,0.920957215373459,0.7113900410214716
8,0.6608315098468271,0.6608315098468271,0.9445660102115244,0.7593297637070227
9,0.6118881118881119,0.6118881118881119,0.9286713286713286,0.7141730223994406
10,0.6304654442877292,0.6304654442877292,0.9308885754583921,0.7299766489670764
11,0.6386315039201711,0.6386315039201711,0.9351389878831077,0.7361867231561487
12,0.6310888252148997,0.6310888252148997,0.9398280802292264,0.7331393493309797
13,0.6288433382137628,0.6288433382137628,0.9304538799414348,0.7260761104097146
14,0.6375791695988741,0.6375791695988741,0.9296270232230823,0.7343394320333564
15,0.6085106382978723,0.6085106382978723,0.9241134751773049,0.7094800721909098
16,0.624715693707354,0.624715693707354,0.9226686884003032,0.7238086337342909
17,0.6288384512683578,0.6288384512683578,0.9299065420560748,0.7284478008320093
18,0.6441005802707931,0.6441005802707931,0.9406834300451322,0.7446700757538023
19,0.6490618485059069,0.6490618485059069,0.9277275886031967,0.746508743208761
20,0.6324614352783366,0.6324614352783366,0.9309188464118041,0.7326145833008225
21,0.6219336219336219,0.6219336219336219,0.9155844155844156,0.7177598078555566
22,0.6508795669824087,0.6508795669824087,0.9248985115020297,0.7442804333947263
23,0.6153314917127072,0.6153314917127072,0.9067679558011049,0.7140897797364366
24,0.6418870621872766,0.6418870621872766,0.9249463902787706,0.7361877726540907
25,0.6151624548736462,0.6151624548736462,0.9133574007220217,0.7134970280166214
26,0.64421768707483,0.64421768707483,0.9346938775510204,0.7426403209887947
27,0.6353741496598639,0.6353741496598639,0.9401360544217687,0.7384086490975394
28,0.656614119259767,0.656614119259767,0.9335161069225497,0.7506990889680019
29,0.6556291390728477,0.6556291390728477,0.9345106696100074,0.7515162209266859
30,0.6695526695526696,0.6695526695526696,0.9321789321789322,0.7576160391830521
31,0.6577540106951871,0.6577540106951871,0.9398395721925134,0.7542342796938609
32,0.6229825693996126,0.6229825693996126,0.9335054874112331,0.7268598820569814
33,0.638283378746594,0.638283378746594,0.9352861035422343,0.7394029705045854
34,0.6469314079422382,0.6469314079422382,0.9285198555956679,0.7431700765079997
35,0.6128808864265928,0.6128808864265928,0.9383656509695291,0.7215222326315571
36,0.6180048661800487,0.6180048661800487,0.9302514193025142,0.7175007975977716
37,0.6371681415929203,0.6371681415929203,0.9278420694349898,0.736130082746311
38,0.6117411607738492,0.6117411607738492,0.9346230820547031,0.7199130770013992
39,0.6314655172413793,0.6314655172413793,0.9188218390804598,0.7306173152302111
40,0.5957132298595713,0.5957132298595713,0.9209164818920916,0.6990906511689003
41,0.6514726507713885,0.6514726507713885,0.9305750350631136,0.746084227100536
42,0.6439490445859872,0.6439490445859872,0.9528662420382166,0.7468495707759908
43,0.6141995981245814,0.6141995981245814,0.9082384460817147,0.7141644051720438
44,0.5994277539341917,0.5994277539341917,0.927038626609442,0.7034377385177669
45,0.6182432432432432,0.6182432432432432,0.9331081081081081,0.7198018157358913
46,0.6109489051094891,0.6109489051094891,0.9357664233576642,0.7160718046317288
47,0.5990133897110641,0.5990133897110641,0.9189570119802678,0.7047918628026498
48,0.6362425049966689,0.6362425049966689,0.9313790806129247,0.7357128018064611
49,0.607566765578635,0.607566765578635,0.9191394658753709,0.709780110336787
50,0.6101818181818182,0.6101818181818182,0.9134545454545454,0.7087680724543174
51,0.647191011235955,0.647191011235955,0.9205992509363295,0.7410059543553524
52,0.6300287356321839,0.6300287356321839,0.9202586206896551,0.7276339828016986
53,0.6264285714285714,0.6264285714285714,0.9064285714285715,0.7226310353746898
54,0.6139954853273137,0.6139954853273137,0.9157261098570354,0.7135130025549458
55,0.636734693877551,0.636734693877551,0.9231292517006803,0.7334516905225551
56,0.6396731054977711,0.6396731054977711,0.9420505200594353,0.743063073113198
57,0.6445461479786423,0.6445461479786423,0.9122807017543859,0.7374186571357618
58,0.6341637010676157,0.6341637010676157,0.9224199288256227,0.7276544157717176
59,0.6057098765432098,0.6057098765432098,0.9112654320987654,0.7049470983291083
60,0.6122296793437733,0.6122296793437733,0.9209545115585384,0.7168503444056544
61,0.6168294515401953,0.6168294515401953,0.9068369646882044,0.7114440955523762
62,0.6297903109182935,0.6297903109182935,0.9342010122921186,0.7284789230026565
63,0.6110304789550073,0.6110304789550073,0.9172714078374455,0.7112000246979674
64,0.6185852981969486,0.6185852981969486,0.9244105409153953,0.7218795566533261
65,0.64280798348245,0.64280798348245,0.9167240192704749,0.7377702630599933
66,0.6345628415300546,0.6345628415300546,0.9357923497267759,0.7374479902205181
67,0.6541254125412541,0.6541254125412541,0.9564356435643564,0.7570419144125053
68,0.6441677588466579,0.6441677588466579,0.9246395806028833,0.7402875261929575
69,0.5990922844175491,0.5990922844175491,0.9107413010590015,0.7008168297098084
70,0.6375092661230541,0.6375092661230541,0.9117865085248332,0.7319353107724683
71,0.6384839650145773,0.6384839650145773,0.9052478134110787,0.7303644603275141
72,0.6203090507726269,0.6203090507726269,0.9300956585724798,0.7210143127447711
73,0.6251768033946252,0.6251768033946252,0.9214992927864215,0.7260309499254365
74,0.61602787456446,0.61602787456446,0.9317073170731708,0.7221183281071267
75,0.6611950098489823,0.6611950098489823,0.9415627051871307,0.7596406089249823
76,0.6324503311258278,0.6324503311258278,0.9456953642384106,0.7372367348542818
77,0.6338028169014085,0.6338028169014085,0.9349429912810194,0.734891189582195
78,0.6361746361746362,0.6361746361746362,0.9348579348579349,0.7378328338357939
79,0.6288109756097561,0.6288109756097561,0.9070121951219512,0.723278395196642
80,0.6062407132243685,0.6062407132243685,0.9071322436849926,0.7095649165764408
81,0.6307803468208093,0.6313222543352601,0.9246490503715937,0.7264327960022748
82,0.6132278936017254,0.6137670740474479,0.9185580774365821,0.715168808395149
83,0.6299782766111514,0.6305213613323678,0.9338988310747905,0.7307387596871595
84,0.6529680365296804,0.653457273320287,0.9352343677196907,0.7512138452052626
85,0.6279863481228669,0.6284414106939704,0.9335608646188851,0.7275615264334889
86,0.6453851492019431,0.6453851492019431,0.9430950728660652,0.7451300335176478
87,0.682,0.6824444444444445,0.9417777777777778,0.77366262853511
88,0.5965541995692749,0.5969131371141422,0.9218413496051687,0.7010789489279943
89,0.6191145467322557,0.6193956430077302,0.907003982197236,0.7140147359789155
90,0.6217948717948718,0.6223290598290598,0.9114774114774116,0.7201480487313836
91,0.600151171579743,0.6003401360544217,0.9090810927545622,0.7020079814788018
92,0.620199146514936,0.6199146514935989,0.9118065433854907,0.7176935170769195
93,0.6495726495726496,0.6493973263204033,0.9428007889546351,0.7496920926730133
94,0.6251830161054173,0.6251830161054173,0.924597364568082,0.7245671288909641
95,0.6645021645021645,0.6648240648240649,0.9248436748436749,0.7546437337184336
96,0.6152091254752852,0.6149293165642976,0.9024081115335869,0.7121222690003721
97,0.6109489051094891,0.6116226838854576,0.9050533408197642,0.7092816797220116
98,0.6200772200772201,0.6222516865374008,0.8866615045186473,0.7111011184506202
99,0.6066619418851878,0.608027196049875,0.9180177724472551,0.7114181200247154
100,0.6599713055954088,0.6636706789145814,0.9157040114357187,0.7471132669544209'''))

In [ ]:
import io
fedprox10_full = pd.read_csv(io.StringIO('''round,accuracy,precision,recall,f1_score
1,0.6352201257861635,0.6352201257861635,0.9252271139063591,0.7320928139349434
2,0.6194362017804155,0.6194362017804155,0.908753709198813,0.7146748851521515
3,0.6404657933042213,0.6404657933042213,0.9381368267831149,0.7380013854899914
4,0.637407157326131,0.637407157326131,0.9237002025658338,0.735759611713053
5,0.6221751412429378,0.6221751412429378,0.9194915254237288,0.7226789187336727
6,0.6187806587245971,0.6187806587245971,0.9355290819901892,0.721393436105054
7,0.6423027166882277,0.6423027166882277,0.94372574385511,0.7433335840271611
8,0.6129985228951256,0.6129985228951256,0.9202363367799113,0.7159962745992008
9,0.6247288503253796,0.6247288503253796,0.925524222704266,0.7236305930765353
10,0.6192821956368755,0.6192821956368755,0.94018296973962,0.7248519888675244
11,0.6250950570342205,0.6250950570342205,0.9193916349809886,0.7223299483970103
12,0.6438953488372093,0.6438953488372093,0.9244186046511628,0.7397233310894961
13,0.6582438408085913,0.6582438408085913,0.936197094125079,0.7534405767663839
14,0.6335149863760218,0.6335149863760218,0.9291553133514986,0.7331895321659465
15,0.6525597269624573,0.6525597269624573,0.931740614334471,0.7471068290995049
16,0.6138888888888889,0.6138888888888889,0.9125,0.7115389395560199
17,0.6319305277221109,0.6319305277221109,0.9418837675350702,0.7321155648673368
18,0.5802650957290133,0.5802650957290133,0.9189985272459499,0.6893936852013595
19,0.6244541484716157,0.6244541484716157,0.9264919941775837,0.7278319776011898
20,0.6484268125854993,0.6484268125854993,0.9274965800273598,0.7451811565891912
21,0.6135895676046671,0.6135895676046671,0.9142072752230611,0.7130532835501735
22,0.6175438596491228,0.6175438596491228,0.9298245614035088,0.721006008916593
23,0.6266760762173607,0.6266760762173607,0.9237826393789696,0.7245986539406448
24,0.6334716459197787,0.6334716459197787,0.9287690179806363,0.732204323071147
25,0.6256983240223464,0.6256983240223464,0.9120111731843575,0.7220633433670038
26,0.635719347980156,0.635719347980156,0.9298369950389794,0.7337539477629247
27,0.6037344398340249,0.6037344398340249,0.9232365145228216,0.7086022886188938
28,0.6259600614439325,0.6259600614439325,0.9170506912442397,0.7216888487844308
29,0.6340277777777777,0.6340277777777777,0.9402777777777778,0.7375089643260347
30,0.6097912167026638,0.6097912167026638,0.9150467962562995,0.7113912453962741
31,0.6334776334776335,0.6334776334776335,0.9227994227994228,0.7319397303686249
32,0.6403631284916201,0.6403631284916201,0.9210893854748603,0.7353879179345368
33,0.6423562412342216,0.6423562412342216,0.9312762973352033,0.7374509725792521
34,0.6167247386759582,0.6167247386759582,0.9240418118466899,0.7167072392691367
35,0.6313993174061433,0.6313993174061433,0.9262798634812287,0.7302797880285562
36,0.636731937879811,0.636731937879811,0.9196488858879136,0.729139803064167
37,0.6428571428571429,0.6428571428571429,0.926490984743412,0.7385977495654888
38,0.6209150326797386,0.6209150326797386,0.9273783587509078,0.7188621348331964
39,0.6192439862542956,0.6192439862542956,0.9360824742268041,0.7250219007354339
40,0.625797306874557,0.625797306874557,0.9326718639262934,0.726947955421464
41,0.6546463245492372,0.6546463245492372,0.9292649098474342,0.7480709445766638
42,0.6297602256699577,0.6297602256699577,0.9280677009873061,0.7297184836729691
43,0.6216783216783217,0.6216783216783217,0.920979020979021,0.7208550612167877
44,0.6173974540311173,0.6173974540311173,0.9222065063649222,0.7178954860150599
45,0.6290322580645161,0.6290322580645161,0.9381720430107527,0.732588918015197
46,0.638041163946061,0.638041163946061,0.9368346344925479,0.7401123938859303
47,0.6302765647743813,0.6302765647743813,0.9301310043668122,0.7308143108546036
48,0.6297653958944281,0.6297653958944281,0.9112903225806451,0.7232766751810153
49,0.6234817813765182,0.6234817813765182,0.9068825910931174,0.7171574107621719
50,0.6352459016393442,0.6352459016393442,0.9296448087431693,0.7335464938889255
51,0.6337644656228727,0.6337644656228727,0.9455411844792376,0.7373439618071912
52,0.6345588235294117,0.6345588235294117,0.9301470588235294,0.7315331461509448
53,0.6341798215511325,0.6341798215511325,0.9245024021962938,0.7319370839409248
54,0.6260623229461756,0.6260623229461756,0.9298866855524079,0.7287077876599269
55,0.6458944281524927,0.6458944281524927,0.9310850439882697,0.7417143521375494
56,0.6374045801526718,0.6374045801526718,0.9282442748091603,0.7353962126553031
57,0.6415094339622641,0.6415094339622641,0.9210342417889588,0.7378056235060987
58,0.6047345767575323,0.6047345767575323,0.9074605451936872,0.7062985548435857
59,0.6416610850636303,0.6416610850636303,0.9263228399196249,0.7387983386089024
60,0.6281669150521609,0.6281669150521609,0.9277198211624441,0.7277700119972049
61,0.6202723146747352,0.6202723146747352,0.9251134644478064,0.7195405270480908
62,0.6130551816958277,0.6130551816958277,0.923956931359354,0.7142671479981981
63,0.6036363636363636,0.6036363636363636,0.9134545454545454,0.7048154799170311
64,0.6177304964539007,0.6177304964539007,0.9106382978723404,0.7161780928687224
65,0.6515937731653076,0.6515937731653076,0.9340252038547072,0.7453428016728532
66,0.6199442119944212,0.6199442119944212,0.9198047419804742,0.7195701443012064
67,0.6083761939750184,0.6083761939750184,0.919911829537105,0.7111669450958763
68,0.6305160807778609,0.6305160807778609,0.9296933433059088,0.7319633876985645
69,0.6049122807017544,0.6049122807017544,0.9129824561403509,0.7091095986538803
70,0.6283783783783784,0.6283783783783784,0.9309309309309309,0.7305750008263341
71,0.650916784203103,0.650916784203103,0.9337094499294781,0.7446775090002311
72,0.6041666666666666,0.6041666666666666,0.9340277777777778,0.710145627805391
73,0.6319942611190817,0.6319942611190817,0.921090387374462,0.7299236989695088
74,0.6483739837398373,0.6483739837398373,0.948509485094851,0.747985461896215
75,0.6321506154960174,0.6321506154960174,0.9348298334540188,0.7329175906017621
76,0.6258692628650904,0.6258692628650904,0.9061196105702365,0.7220010486259573
77,0.6238805970149254,0.6238805970149254,0.9164179104477612,0.7231329050314479
78,0.6336705202312138,0.6336705202312138,0.916907514450867,0.7274664021331702
79,0.6371621621621621,0.6371621621621621,0.9304054054054054,0.7333339260886425
80,0.6448979591836734,0.6448979591836734,0.9292517006802721,0.7408170478614079
81,0.6374228923920493,0.6374228923920493,0.9294037011651817,0.734789418999266
82,0.631367292225201,0.631367292225201,0.9309651474530831,0.7329509677803792
83,0.6075498575498576,0.6075498575498576,0.9131054131054132,0.7096428657347451
84,0.6511976047904192,0.6511976047904192,0.9348802395209581,0.7487266563968009
85,0.6233148295003965,0.6233148295003965,0.9032513877874703,0.7151605853863112
86,0.6321094312455003,0.6321094312455003,0.9301655867530597,0.7311572602309034
87,0.6399735274652548,0.6399735274652548,0.9364659166115156,0.7437957207869933
88,0.6360239162929746,0.6360239162929746,0.9185351270553064,0.733383675224223
89,0.6228932584269663,0.6228932584269663,0.9037921348314607,0.7177863102433254
90,0.632258064516129,0.632258064516129,0.9268817204301075,0.7302790766555565
91,0.6154366960054164,0.6154366960054164,0.9214624238320921,0.7177400831138535
92,0.6576923076923077,0.6576923076923077,0.9358974358974359,0.7523108653723777
93,0.6670992861778067,0.6670992861778067,0.9344581440622972,0.758069901545171
94,0.6308345120226309,0.6308345120226309,0.92998585572843,0.7318910017803761
95,0.6205923836389281,0.6205923836389281,0.919605077574048,0.7209797793840136
96,0.6220760233918129,0.6220760233918129,0.9217836257309941,0.7184187028612031
97,0.6063756063756064,0.6067221067221067,0.9155405405405406,0.7115182552509711
98,0.5921696574225123,0.5921696574225123,0.8915171288743883,0.6886745646652133
99,0.658751696065129,0.6595997286295794,0.9286562318278737,0.7518369718215009
100,0.6345628415300546,0.6345628415300546,0.9439890710382514,0.7356911327774138'''))

In [ ]:
strategies_full = {'FedAvg': fedavg_full, 'FedProx_mu0.1': fedprox01_full, 'FedProx_mu0.5': fedprox05_full, 'FedProx_mu1.0': fedprox10_full}
mus = {'FedAvg': 0.0, 'FedProx_mu0.1': 0.1, 'FedProx_mu0.5': 0.5, 'FedProx_mu1.0': 1.0}

rows = []
for label, d in strategies_full.items():
    best = d.loc[d.f1_score.idxmax()]
    rows.append({'strategy': label, 'mu': mus[label], 'best_f1': round(best.f1_score, 4), 'best_round': int(best['round']),
                 'mean_f1': round(d.f1_score.mean(), 4), 'std_dev': round(d.f1_score.std(), 4)})
summary_full = pd.DataFrame(rows)
summary_full

### Comparison against the paper's Table 6

| Strategy | Best F1 (paper) | Best F1 (this run) | Best round (paper) | Best round (this run) | Std Dev (paper) | Std Dev (this run) |
|---|---|---|---|---|---|---|
| FedAvg | 0.7584 | 0.7710 | 70 | 23 | 0.0249 | 0.0150 |
| FedProx mu=0.1 | 0.7526 | 0.7638 | 89 | 37 | 0.0242 | 0.0161 |
| **FedProx mu=0.5** | **0.7628** | **0.7737** | 88 | 87 | 0.0205 | 0.0162 |
| FedProx mu=1.0 | 0.7555 | 0.7581 | 80 | 93 | **0.0152** | **0.0129** |

**What replicates:** FedProx (mu=0.5) achieves the highest peak F1-Score among the four strategies in both the paper and this independent run — the paper's central claim. FedProx (mu=1.0) achieves the lowest standard deviation (most stable training) in both runs as well, confirming the stability-performance trade-off the paper describes.

**What diverges:** absolute F1 values in this run are systematically ~1-1.5 percentage points higher than the paper across all four strategies, and the best round differs substantially per strategy. Most notably, the paper reports FedAvg as the *least* stable strategy (highest std dev, 0.0249) while in this run FedAvg was among the *most* stable (0.0150). This is plausible run-to-run variance inherent to Flower/Ray's client-sampling and actor-scheduling non-determinism — it does not reproduce bit-for-bit even with a fixed `torch.manual_seed`, since client selection order and cross-process floating-point summation order are not fully pinned by that seed alone. This divergence is reported here transparently rather than adjusted to match the paper.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle('Federated strategies — F1-Score, Accuracy, Precision, Recall over 100 rounds (full reproduction)', y=1.03)
colors = {'FedAvg': 'darkorange', 'FedProx_mu0.1': 'seagreen', 'FedProx_mu0.5': 'steelblue', 'FedProx_mu1.0': 'firebrick'}
for ax, metric in zip(axes, ['accuracy', 'precision', 'recall', 'f1_score']):
    for label, d in strategies_full.items():
        ax.plot(d['round'], d[metric].rolling(5, min_periods=1).mean(), label=label, color=colors[label], linewidth=1.6)
    ax.set_title(metric.replace('_', ' ').title()); ax.set_xlabel('Communication Round'); ax.grid(alpha=0.3)
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(fedprox05_full['round'], fedprox05_full['f1_score'], color='steelblue', linewidth=1.8, label='FedProx (mu=0.5)')
plt.plot(fedavg_full['round'], fedavg_full['f1_score'], color='darkorange', linestyle='--', linewidth=1.6, label='FedAvg')
plt.axhline(0.7628, color='steelblue', linestyle=':', alpha=0.6, label='Paper FedProx peak (0.7628)')
plt.axhline(0.7584, color='darkorange', linestyle=':', alpha=0.6, label='Paper FedAvg peak (0.7584)')
plt.xlabel('Communication Round'); plt.ylabel('F1-Score')
plt.title('F1-Score: FedProx (mu=0.5) vs FedAvg — this reproduction vs. paper peaks')
plt.legend(fontsize=9); plt.grid(alpha=0.3); plt.show()

## 4. Code Review Notes

While reproducing this pipeline, two real gaps were found and fixed:

- **`requirements.txt` was missing `ray`.** `flwr.simulation.start_simulation` requires the `flwr[simulation]` extra (which depends on `ray`), and a plain `pip install -r requirements.txt` fails with `ImportError: Unable to import module 'ray'` before any training starts. Fixed by adding `ray==2.6.3` to the repository's `requirements.txt` (validated on Python 3.11).
- **The repo's pinned `flwr==1.7.0`/`ray==2.6.3` cannot install on Python 3.12** (e.g. Kaggle's current default environment) — `ray==2.6.3` has no Python 3.12 wheel. This notebook works around it by installing unpinned/latest `flwr`+`ray` (see Section 0); the repository itself still targets Python 3.10/3.11 per its README, so this is a Kaggle-runtime-specific accommodation rather than a repo fix.

Everything else — filtering thresholds, feature engineering, ID mapping, MinMax scaling, the DNN architecture (Table 3), FedProx's proximal term (Eq. 4), and all Table 2 hyperparameters (learning rate, batch size, local epochs, client fractions, `min_fit_clients`, seed) — matched the paper's description exactly.

## 5. Conclusion

This notebook independently verifies that the code in [rodrigoronner/federated-recommender-education](https://github.com/rodrigoronner/federated-recommender-education) implements the methodology described in the JEDM submission faithfully. The centralized XGBoost baseline reproduces the paper's numbers almost exactly (within ~0.1pp). The federated experiments reproduce the paper's central qualitative claims — FedProx (mu=0.5) gives the best peak F1-Score, and FedProx (mu=1.0) gives the most stable training — while showing that exact per-round numbers (and which strategy is *least* stable) carry meaningful run-to-run variance inherent to federated simulation with Flower/Ray, worth disclosing for anyone attempting to reproduce Table 6 exactly.

**Citation:**
```
Tertulino, R., & Almeida, R. (2025). Privacy-preserving personalization in education: A federated recommender
system for student performance prediction. arXiv preprint arXiv:2509.10516.
```